In [1]:
import os
import json
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import svm
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GridSearchCV
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.under_sampling import RandomUnderSampler, NearMiss
from imblearn.over_sampling import SMOTE, RandomOverSampler
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.layers import GRU
from sklearn.feature_extraction.text import TfidfVectorizer

# Set random seed for NumPy
np.random.seed(42)

# Set random seed for TensorFlow v1
tf.random.set_seed(42)

# Read the data
file_path = 'domain1_train_data.json'
raw_data_domain1 = pd.read_json(file_path, lines=True)
file_path = 'domain2_train_data.json'
train_data_domain2 = pd.read_json(file_path, lines=True)
file_path = 'test_data.json'
test_data = pd.read_json(file_path, lines=True)

# Split into train and validation data
label_column = 'label'
train_data_domain1, validation_data_domain1 = train_test_split(raw_data_domain1, test_size=0.1, random_state=42, stratify=raw_data_domain1[label_column])
train_domain1_y = np.array(train_data_domain1[label_column])

# Create a vectorizer
vectorizer = CountVectorizer()

# Fit the vectorizer to all the text data
all_text = pd.concat([train_data_domain1['text'].apply(str),train_data_domain2['text'].apply(str)])
vectorizer.fit(all_text)

# Transform the text data into vector in form of a 2D array
train_domain1_x = vectorizer.transform(train_data_domain1['text'].apply(str)).toarray()

validation_domain1_y = np.array(validation_data_domain1[label_column])
validation_domain1_x = vectorizer.transform(validation_data_domain1['text'].apply(str)).toarray()

train_domain2_y = np.array(train_data_domain2[label_column])
train_domain2_x = vectorizer.transform(train_data_domain2['text'].apply(str)).toarray()

In [2]:
# Instantiate Random Forest Classifier for domain 1 data
rf_classifier_1 = RandomForestClassifier(
    bootstrap=False,
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=5,
    n_estimators=200,
    random_state=42
)

# Train the model
rf_classifier_1.fit(train_domain1_x, train_domain1_y)

# Predict the labels of the validation set
validation_predictions_rf = rf_classifier_1.predict(validation_domain1_x)

# Calculate the precision, recall, and F1 score of the model
precision_rf = precision_score(validation_domain1_y, validation_predictions_rf)
recall_rf = recall_score(validation_domain1_y, validation_predictions_rf)
f1_rf = f1_score(validation_domain1_y, validation_predictions_rf)

# Print the precision, recall, and F1 score of the model
print(f'Precision: {precision_rf}')
print(f'Recall: {recall_rf}')
print(f'F1 Score: {f1_rf}')

Precision: 0.7954545454545454
Recall: 0.84
F1 Score: 0.8171206225680934


In [3]:
# Train a list of model for domain 2 data
# Separate minority and majority class instances
minority_class = train_data_domain2[train_data_domain2[label_column] == 1]
majority_class = train_data_domain2[train_data_domain2[label_column] == 0]

# Shuffle the majority class instances
majority_class = majority_class.sample(frac=1, random_state=42)

# Calculate the number of subsets (models) based on the ratio of majority to minority class instances
num_subsets = len(majority_class) // len(minority_class)

# Initialize a list to hold the trained models
models_rf_domain_2 = []

# Loop through each subset, pair minority class with a portion of the majority class, and train a model
for i in range(num_subsets):
    # Select a subset of the majority class
    subset_majority = majority_class[i * len(minority_class):(i + 1) * len(minority_class)]
    
    # Combine minority and majority class instances
    subset_data = pd.concat([minority_class, subset_majority])

    # Shuffle the subset data
    subset_data = subset_data.sample(frac=1, random_state=42)
    
    # Split the subset data into train and validation sets
    subset_x, validation_domain2_x, subset_y, validation_domain2_y = train_test_split(subset_data['text'], subset_data[label_column], test_size=0.1, random_state=42, stratify=subset_data[label_column])
    
    # Split data into features and labels
    subset_x = vectorizer.transform(subset_x.apply(str)).toarray()
    validation_domain2_x = vectorizer.transform(validation_domain2_x.apply(str)).toarray()
    
    # Train a classifier
    rf_classifier = RandomForestClassifier(
        bootstrap=False,
        max_depth=None,
        max_features='sqrt',
        min_samples_leaf=1,
        min_samples_split=5,
        n_estimators=200,
        random_state=42
    )

    rf_classifier.fit(subset_x, subset_y)

    validation_predictions_rf = rf_classifier.predict(validation_domain2_x)

    # Calculate the precision, recall, and F1 score of the model
    precision_rf = precision_score(validation_domain2_y, validation_predictions_rf)
    recall_rf = recall_score(validation_domain2_y, validation_predictions_rf)
    f1_rf = f1_score(validation_domain2_y, validation_predictions_rf)

    # Print the precision, recall, and F1 score of the model
    print(f'Precision: {precision_rf}')
    print(f'Recall: {recall_rf}')
    print(f'F1 Score: {f1_rf}')

    print("1", np.sum(validation_predictions_rf == 1))
    print("0", np.sum(validation_predictions_rf == 0))
    
    # Append trained model to the list
    models_rf_domain_2.append(rf_classifier)


Precision: 0.8089887640449438
Recall: 0.96
F1 Score: 0.8780487804878049
1 178
0 122
Precision: 0.7944444444444444
Recall: 0.9533333333333334
F1 Score: 0.8666666666666666
1 180
0 120
Precision: 0.7857142857142857
Recall: 0.9533333333333334
F1 Score: 0.8614457831325301
1 182
0 118
Precision: 0.8033707865168539
Recall: 0.9533333333333334
F1 Score: 0.8719512195121952
1 178
0 122
Precision: 0.8011363636363636
Recall: 0.94
F1 Score: 0.8650306748466258
1 176
0 124
Precision: 0.7724867724867724
Recall: 0.9733333333333334
F1 Score: 0.8613569321533924
1 189
0 111
Precision: 0.772972972972973
Recall: 0.9533333333333334
F1 Score: 0.8537313432835821
1 185
0 115


In [4]:
# train an binary classifier to split domain1 and domain2
domain1_y = np.zeros(raw_data_domain1.shape[0])
# Calculate the size of each subset
subset_size = raw_data_domain1.shape[0]

# Shuffle the domain2 data and split it into two parts of same size as domain1 data
train_data_domain2 = train_data_domain2.sample(frac=1, random_state=42)
domain2_part1 = train_data_domain2.iloc[:subset_size]
domain2_part2 = train_data_domain2.iloc[subset_size:]

# Sample the same number of samples from each part and shuffle them
domain2_subset1 = domain2_part1.sample(n=subset_size, random_state=42)
domain2_subset2 = domain2_part2.sample(n=subset_size, random_state=42)

# Pair the domain1 and first subset of domain2 data
domain2_y = np.ones(domain2_subset1.shape[0])
train_domain2_x = vectorizer.transform(domain2_subset1['text'].apply(str)).toarray()
domain_y = np.concatenate((domain1_y, domain2_y), axis=0)
domain_x = np.concatenate((vectorizer.transform(raw_data_domain1['text'].apply(str)).toarray(), train_domain2_x), axis=0)

early_stopping = callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)

# Train the MLP domain classification model
domain_model = models.Sequential([
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

domain_model.compile(optimizer='adam',
                loss='binary_crossentropy',
                metrics=['accuracy'])

train_domain_x, validation_domain_x, train_domain_y, validation_domain_y = train_test_split(domain_x, domain_y, test_size=0.1, random_state=42, stratify=domain_y)

checkpoint_filepath_domain = 'best_model_domain.keras'

model_checkpoint_callback_domain = ModelCheckpoint(
    filepath=checkpoint_filepath_domain,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

history_2 = domain_model.fit(train_domain_x, train_domain_y, epochs=50, batch_size=256, validation_data=(validation_domain_x,validation_domain_y), callbacks=[model_checkpoint_callback_domain, early_stopping])

# Load the best model saved during training for the second model
best_model_domain = models.load_model(checkpoint_filepath_domain)

# Pair the domain1 data and second subset of domain2 data
domain2_y = np.ones(domain2_subset2.shape[0])
train_domain2_x = vectorizer.transform(domain2_subset2['text'].apply(str)).toarray()
domain_y = np.concatenate((domain1_y, domain2_y), axis=0)
domain_x = np.concatenate((vectorizer.transform(raw_data_domain1['text'].apply(str)).toarray(), train_domain2_x), axis=0)

# Train the MLP domain classification model
domain_model = models.Sequential([
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

domain_model.compile(optimizer='adam',
                loss='binary_crossentropy',
                metrics=['accuracy'])

train_domain_x, validation_domain_x, train_domain_y, validation_domain_y = train_test_split(domain_x, domain_y, test_size=0.1, random_state=42, stratify=domain_y)

checkpoint_filepath_domain = 'best_model_domain_2.keras'

model_checkpoint_callback_domain = ModelCheckpoint(
    filepath=checkpoint_filepath_domain,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

history_2 = domain_model.fit(train_domain_x, train_domain_y, epochs=50, batch_size=256, validation_data=(validation_domain_x,validation_domain_y), callbacks=[model_checkpoint_callback_domain, early_stopping])

# Load the best model saved during training for the second model
best_model_domain2 = models.load_model(checkpoint_filepath_domain)

Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 326ms/step - accuracy: 0.9303 - loss: 7.9997
Epoch 1: val_accuracy improved from -inf to 0.99200, saving model to best_model_domain.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 20s 451ms/step - accuracy: 0.9316 - loss: 7.9248 - val_accuracy: 0.9920 - val_loss: 2.2924
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 323ms/step - accuracy: 0.9961 - loss: 1.8164
Epoch 2: val_accuracy improved from 0.99200 to 0.99800, saving model to best_model_domain.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 13s 359ms/step - accuracy: 0.9961 - loss: 1.8065 - val_accuracy: 0.9980 - val_loss: 0.8254
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 322ms/step - accuracy: 0.9920 - loss: 0.7214
Epoch 3: val_accuracy did not improve from 0.99800
36/36 ━━━━━━━━━━━━━━━━━━━━ 12s 331ms/step - accuracy: 0.9921 - loss: 0.7180 - val_accuracy: 0.9970 - val_loss: 0.3787
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 323ms/step - accuracy: 0.9946 - loss: 0.3688
Epoch 4: val_accuracy did not improve from 0.99800
36/36 ━━━━━

## Generating test data

In [5]:
# Predict the domain of the test data and split into two subsets
test_x = vectorizer.transform(test_data['text'].apply(str)).toarray()
domain_pred = best_model_domain.predict(test_x)
domain_pred_1 = best_model_domain2.predict(test_x)
domain_pred = np.mean([domain_pred, domain_pred_1], axis=0)
domain_pred = np.where(domain_pred > 0.5, 1, 0)
domain1_index = np.where(domain_pred == 0)
domain2_index = np.where(domain_pred == 1)
test_domain_1 = test_data[domain_pred == 0]
test_domain_2 = test_data[domain_pred == 1]
print('domain1:', len(test_domain_1))
print('domain2:', len(test_domain_2))

125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
domain1: 1995
domain2: 2005


In [6]:
# Prediction for segmented domain1 test data
test_predict_domain1 = rf_classifier_1.predict(vectorizer.transform(test_domain_1['text'].apply(str)).toarray())

# Prediction for segmented domain2 test data
predictions_per_model_domain2 = []
for rf_model in models_rf_domain_2:
    predictions = rf_model.predict(vectorizer.transform(test_domain_2['text'].apply(str)).toarray())
    predictions_per_model_domain2.append(predictions)

# Calculate the average of the domain2 ensembled predictions
average_predictions_domain2 = np.mean(predictions_per_model_domain2, axis=0)
average_predictions_domain2 = np.where(average_predictions_domain2 > 0.99, 1, 0)

test_predict_domain2 = average_predictions_domain2

test_predict_domain1 = np.where(test_predict_domain1 > 0.5, 1, 0)
print('1:', np.sum(test_predict_domain1 == 1))
print('0:', np.sum(test_predict_domain1 == 0))
test_predict_domain2 = np.where(test_predict_domain2 > 0.5, 1, 0)
print('1:', np.sum(test_predict_domain2 == 1))
print('0:', np.sum(test_predict_domain2 == 0))

1: 1025
0: 970
1: 982
0: 1023


In [7]:
# Initialize the 'label' column with default values
test_data['label'] = 0

# Update the 'label' column with predictions for domain 1
test_data.loc[domain1_index[0], 'label'] = test_predict_domain1.flatten()

# Update the 'label' column with predictions for domain 2
test_data.loc[domain2_index[0], 'label'] = test_predict_domain2.flatten()

# Convert the 'label' column to integer type
test_data['label'] = test_data['label'].astype(int)

print('1:', np.sum(test_data['label'] == 1))
print('0:', np.sum(test_data['label'] == 0))

# Save the predictions to a CSV file
test_data[['id', 'label']].to_csv('sample.csv', index=False)

1: 2007
0: 1993
